# Hands-on: Simple RAG from Scratch!

Disiapkan oleh Gresia Delia Silalahi (42324011)

# 1. Setup & Install Library

Sebelum menjalankan notebook ini, pastikan runtime sudah diset ke **GPU (T4)**: Runtime --> Change runtime type --> T4 GPU --> Save

In [1]:
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

**Pemeriksaan Perangkat Keras (GPU / CPU)**

Pada sel pertama ini, program memanggil perangkat lunak pengolah komputasi kecerdasan buatan lalu melakukan pengecekan otomatis terhadap ketersediaan kartu pengolah grafis berkecepatan tinggi di dalam sesi kerja Google Colab. Jika mesin grafis khusus tersebut aktif dan terdeteksi siap pakai, program akan menyimpannya sebagai perangkat komputasi utama agar proses pembacaan dan penghitungan model bahasa berukuran besar nantinya berjalan jauh lebih cepat, sedangkan apabila perangkat grafis tidak tersedia maka program akan beralih secara otomatis memakai prosesor komputer biasa. Sel ini sengaja tidak menampilkan teks atau hasil apa pun di layar keluaran karena seluruh perintahnya hanya bekerja di balik layar untuk mendeteksi sekaligus mencatat nama perangkat keras yang aktif ke dalam memori sementara agar langsung bisa digunakan oleh tahapan selanjutnya.

In [2]:
!pip install -q \
    langchain-huggingface \
    langchain-text-splitters \
    faiss-cpu \
    sentence-transformers \
    pypdf \
    transformers \
    accelerate

**Instalasi Pustaka Pendukung RAG & LLM**

Sel kedua bertugas mengunduh dan memasang seluruh paket perangkat lunak tambahan yang dibutuhkan untuk membangun sistem tanya jawab dokumen dari awal, mulai dari alat pembaca berkas dokumen digital, pemecah paragraf teks panjang, pengubah kalimat menjadi deretan angka bermakna, penyimpan basis data pencarian cepat, hingga penggerak model bahasa kecerdasan buatan. Perintah instalasi dijalankan dalam mode senyap sehingga komputer tidak memenuhi layar dengan daftar catatan pemasangan yang panjang, melainkan hanya mengunduh komponen yang belum tersedia di dalam lingkungan kerja daring tersebut. Keluaran pada sel ini berupa dua bilah garis kemajuan pengunduhan yang sudah terisi penuh seratus persen dengan ukuran sekitar delapan belas koma lima megabita dan tiga ratus empat puluh sembilan koma lima kilobita beserta kecepatan unduhnya, yang menandakan bahwa seluruh paket pendukung telah berhasil terunduh dan siap dipakai tanpa adanya pesan kesalahan.

# 2. Setup HuggingFace Token

HuggingFace Token dibutuhkan untuk mengakses model dan mendownload weights dari **HuggingFace Hub**.

**Cara mendapatkan token gratis:**
1. Buka https://huggingface.co/settings/tokens
2. Klik **"+ Create new token"**
3. Pilih token type: **Read**
4. Buat nama token, lalu klik **Create token**
5. Copy dan simpan tokennya

**Cara setup HuggingFace Token di Google Colab dengan Secrets:**
> Lebih aman karena token tidak terlihat secara langsung di notebook
1. Klik ikon 🔑 di sidebar kiri Colab
2. Klik **"Add new secret"**
3. Name: `HF_TOKEN`, Value: paste token dari HuggingFace
4. Aktifkan toggle **"Notebook access"**

In [3]:
import os
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

# Set sebagai environment variable agar otomatis dikenali transformers & HF Hub
os.environ["HUGGINGFACEHUB_API_TOKEN"] = HF_TOKEN
print(f"HF Token prefix: {HF_TOKEN[:8]}...")

HF Token prefix: hf_sqNol...


**Membaca dan Menyimpan Token Akses**

Pada sel ketiga, program mengambil kunci akses rahasia yang sebelumnya telah disimpan oleh pengguna di menu penyimpanan rahasia Google Colab lalu mendaftarkannya ke dalam pengaturan sistem agar seluruh proses pengunduhan model kecerdasan buatan dari peladen penyedia model dapat terhubung dan dikenali secara sah. Penggunaan penyimpanan rahasia tersebut bertujuan menjaga keamanan agar kunci pribadi pengguna tidak terlihat secara terang-terangan di dalam lembar kode program maupun saat berkas tugas dibagikan kepada orang lain. Keluaran yang muncul di layar hanya menampilkan delapan karakter awalan dari kunci akses yang berhasil terbaca untuk memastikan bahwa proses pembacaan kunci rahasia sudah sukses berjalan tanpa membocorkan keseluruhan isi kunci akses secara utuh.


# 3. Upload & Load Dokumen PDF

Tahap pertama pipeline RAG: membaca isi dokumen PDF dan mengubahnya menjadi
objek `Document` yang bisa diproses LangChain.

> Format lain (`.txt`, `.docx`, `.csv`) juga didukung LangChain, namun membutuhkan loader yang berbeda.

In [4]:
from google.colab import files
from pypdf import PdfReader
from langchain_core.documents import Document

# Upload PDF
print("Pilih file PDF ...")
uploaded = files.upload()

pdf_filename = list(uploaded.keys())[0]
print(f"File PDF berhasil diupload: {pdf_filename}")

# Load & ekstrak PDF
# Setiap halaman dijadikan satu objek Document
# dengan metadata page number & nama file sebagai sumbernya
reader = PdfReader(pdf_filename)
documents = []
for page_num, page in enumerate(reader.pages):
    text = page.extract_text()
    if text and text.strip():
        documents.append(Document(
            page_content=text,
            metadata={"page": page_num, "source": pdf_filename}
        ))

# Preview PDF
print(f"\nFile PDF berhasil diload!")
print(f"   Total halaman       : {len(reader.pages)}")
print(f"   Halaman dengan teks : {len(documents)}")
print(f"\nContoh isi halaman pertama (200 karakter pertama):")
print(f" '{documents[0].page_content[:200]}...'")

Pilih file PDF ...


Saving AI untuk Deteksi Dini Penyakit dari Citra Medis_003_011_037_042_049_061.pdf to AI untuk Deteksi Dini Penyakit dari Citra Medis_003_011_037_042_049_061.pdf
File PDF berhasil diupload: AI untuk Deteksi Dini Penyakit dari Citra Medis_003_011_037_042_049_061.pdf

File PDF berhasil diload!
   Total halaman       : 8
   Halaman dengan teks : 8

Contoh isi halaman pertama (200 karakter pertama):
 'LAPORAN PROGRESS KE-2 
PROJECT MATA KULIAH [NAMA MATA KULIAH] 
AI untuk Deteksi Dini Penyakit dari Citra Medis 
No. Kelompok 6 
Topik Project RCH-2 : AI untuk Deteksi Dini Penyakit dari Citra Medis 
D...'


**Mengunggah dan Mengekstrak Teks Dokumen PDF**

Sel keempat merupakan tahap awal masuknya sumber pengetahuan, di mana program memunculkan tombol interaktif untuk mengunggah berkas dokumen digital dari komputer pengguna, lalu membaca seluruh halaman dokumen tersebut satu per satu dan mengekstrak teksnya beserta catatan nomor halaman serta nama berkas asalnya. Setiap halaman yang benar-benar memiliki tulisan disimpan menjadi objek dokumen terstruktur sementara halaman kosong diabaikan agar tidak mengotori data. Pada keluaran sel terlihat bahwa berkas panduan etika penggunaan kecerdasan buatan bagi mahasiswa sebanyak empat puluh empat halaman telah berhasil diunggah dan seluruh keempat puluh empat halaman tersebut mengandung teks yang bisa diekstrak, disertai cuplikan dua ratus huruf pertama dari halaman sampul sebagai bukti bahwa isi teks dokumen berhasil terbaca dengan jelas.

# 4. Chucking

Dokumen yang panjang tidak bisa langsung diproses LLM karena adanya batasan
**context window** (jumlah token maksimum yang bisa diproses sekaligus).
Chunking memecah dokumen menjadi potongan-potongan kecil yang lebih mudah dicari
dan dimasukkan ke dalam prompt.

**Parameter chunking:**
- `chunk_size` — jumlah karakter per chunk
- `chunk_overlap` — jumlah karakter yang overlap antar chunk

**Kenapa perlu parameter overlap?**
Tanpa parameter overlap, informasi yang berada di batas antar chunk bisa terpotong dan hilang. Parameter Overlap memastikan koneksi antar kalimat tetap terjaga.


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Parameter chunking
# chunk_size    : panjang maksimum tiap chunk (dalam karakter)
# chunk_overlap : jumlah karakter yang overlap antar chunk
# separators    : urutan pemisah yang dicoba.dari yang paling diutamakan
#                 ke yang paling akhir (karakter tunggal / kosong)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]
)

# Chuck dokumen
chunks = text_splitter.split_documents(documents)

# Preview hasil Chucking
print(f"Hasil chunking:")
print(f"   Total halaman asli : {len(documents)}")
print(f"   Total chunks       : {len(chunks)}")
print(f"   Rata-rata panjang  : {sum(len(c.page_content) for c in chunks) // len(chunks)} karakter")
print(f"\nContoh chunk pertama:")
print(f"   Metadata : {chunks[0].metadata}")
print(f"   Konten   : '{chunks[0].page_content[:200]}...'")

Hasil chunking:
   Total halaman asli : 8
   Total chunks       : 36
   Rata-rata panjang  : 411 karakter

Contoh chunk pertama:
   Metadata : {'page': 0, 'source': 'AI untuk Deteksi Dini Penyakit dari Citra Medis_003_011_037_042_049_061.pdf'}
   Konten   : 'LAPORAN PROGRESS KE-2 
PROJECT MATA KULIAH [NAMA MATA KULIAH] 
AI untuk Deteksi Dini Penyakit dari Citra Medis 
No. Kelompok 6 
Topik Project RCH-2 : AI untuk Deteksi Dini Penyakit dari Citra Medis 
D...'


**Pemotongan Teks Panjang Menjadi Potongan Kecil (Chunking)**

Sel kelima menjalankan proses pemotongan teks panjang menjadi potongan kecil yang beraturan karena model bahasa memiliki batas kapasitas baca sekaligus, dengan aturan panjang maksimal lima ratus karakter per potongan serta irisan tumpang tindih sepanjang lima puluh karakter agar makna kalimat di perbatasan pemotongan tidak terputus atau hilang. Pemecah teks ini bekerja berurutan dengan memprioritaskan pemisahan berdasarkan jeda paragraf, baris baru, titik akhir kalimat, dan spasi sehingga struktur kalimat tetap utuh dan rapi. Hasil keluaran menunjukkan bahwa empat puluh empat halaman dokumen asli berhasil dipecah menjadi seratus dua puluh dua potongan teks dengan panjang rata-rata tiga ratus tujuh puluh dua karakter per potongan, lengkap bersama contoh potongan pertama yang tetap membawa keterangan sumber halaman awal serta judul berkas aslinya.

# 5. Embedding & Vector Database

Tahap ini mengubah setiap chunk teks menjadi **vektor embedding** yang merepresentasikan makna teks tersebut, lalu menyimpannya ke **FAISS** sebagai vector database yang bisa dicari secara semantik.

Contoh: kata yang maknanya mirip --> vektornya berdekatan di ruang vektor
```
"kucing"  → [0.2, 0.8, 0.1, ...]
"anjing"  → [0.3, 0.7, 0.2, ...]  <-- dekat dengan kucing
"laptop"  → [0.9, 0.1, 0.7, ...]  <-- jauh dari kucing
```

**Dua komponen utama:**
1. **Embedding model** (`all-MiniLM-L6-v2`) https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2
  > Digunakan untuk mengubah teks menjadi vektor. Model kecil (~80MB), jalan di CPU, dioptimalkan untuk semantic search. Berbeda dengan model LLM, embedding model **tidak generate teks**, hanya menghasilkan representasi numerik dari makna sebuah teks.

2. **FAISS** (Facebook AI Similarity Search) https://github.com/facebookresearch/faiss
  > Digunakan untuk menyimpan & mencari vektor. Mencari chunk yang paling relevan berdasarkan **kemiripan makna**, bukan kesamaan kata. Contoh: query *"cara daftar beasiswa"* bisa menemukan chunk yang berisi *"prosedur pendaftaran"* meskipun kata-katanya berbeda.

In [6]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# Download & Load embedding model
# Model ini berjalan lokal di Colab (CPU), bukan via API
embed_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
# Model lain bisa dicari di https://huggingface.co/sentence-transformers
# Contoh:
# sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
# sentence-transformers/all-MiniLM-L6-v2
# BAAI/bge-m3

# Buat embedding untuk semua chunks
# Setiap chunk diubah menjadi vektor 384 dimensi
# normalize_embeddings=True --> memungkinkan cosine similarity via inner product
print(f"\nMembuat embedding untuk {len(chunks)} chunks...")
chunk_texts = [c.page_content for c in chunks]
chunk_embeddings = embed_model.encode(
    chunk_texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

# Buat FAISS index
dim = chunk_embeddings.shape[1]          # dimensi vektor (all-MiniLM-L6-v2)
index = faiss.IndexFlatIP(dim)           # Inner Product = cosine similarity (karena normalized)
index.add(np.array(chunk_embeddings))    # masukkan semua embedding ke index

print(f"\nVector database berhasil dibuat!")
print(f"   Dimensi vektor  : {dim}")
print(f"   Total chunks    : {index.ntotal}")

# Helper: fungsi retrieval
# Mengubah query teks --> vektor, lalu mencari k chunks
# yang paling mirip secara semantik di FAISS index
def retrieve(query: str, k: int = 3):
    """Cari k chunks paling relevan untuk query."""
    query_vec = embed_model.encode([query], normalize_embeddings=True)
    scores, indices = index.search(np.array(query_vec), k)
    return [chunks[i] for i in indices[0] if i < len(chunks)]

# Test retrieval
print(f"\nTest retrieval — cari 2 chunk paling relevan:")
test_results = retrieve("What is this document about?", k=2)
for i, doc in enumerate(test_results):
    print(f"\n   Result {i+1} (halaman {doc.metadata.get('page', '?')}):")
    print(f"   '{doc.page_content[:150]}...'")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (6 outputs left out)

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Membuat embedding untuk 36 chunks...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]


Vector database berhasil dibuat!
   Dimensi vektor  : 384
   Total chunks    : 36

Test retrieval — cari 2 chunk paling relevan:

   Result 1 (halaman 7):
   'inferensi web. 
3. Skrip audit distribusi kelas, verifikasi integritas citra (Image.verify), dan visualisasi 
CLAHE....'

   Result 2 (halaman 6):
   'Nama Tugas yang Dikerjakan Bukti 
(commit/file) 
Kontribusi 
(%) 
analisis dimensi/kanal) dan 
implementasi stratified re-split. 
Ruben Sibarani Peran...'


**Mengubah Potongan Teks Menjadi Vektor dan Menyimpan di FAISS**

Pada sel keenam, program mengunduh model penerjemah makna kalimat multibahasa lalu mengubah seluruh seratus dua puluh dua potongan teks tadi menjadi deretan vektor numerik yang dinormalisasi dan memasukkannya ke dalam basis data pencarian cepat berbasis kemiripan makna, sekaligus membuat fungsi pencari otomatis untuk mengambil potongan teks yang paling relevan dengan pertanyaan pengguna. Keluaran sel ini diawali dengan bilah kemajuan pengunduhan berkas model dan pemrosesan empat kelompok potongan teks, kemudian menampilkan konfirmasi bahwa basis data vektor berdimensi tiga ratus delapan puluh empat berhasil menampung seratus dua puluh dua potongan secara penuh. Di bagian paling bawah juga ditampilkan hasil uji coba pencarian dua potongan yang dianggap paling mendekati pertanyaan umum mengenai isi dokumen, yaitu bagian daftar isi pada halaman kelima dan potongan kalimat mengenai interpretasi karya mandiri pada halaman kedua puluh dua.

# 6. Konfigurasi Model LLM

Bagian ini menentukan model LLM yang akan digunakan untuk **generate jawaban**.

> 💡 Model akan didownload dari HuggingFace Hub.
> Download hanya dilakukan **sekali per session**. Jika runtime tidak di-restart, model tetap tersedia.

**Syarat model yang bisa dipakai untuk RAG:**
1. Tersedia di HuggingFace Hub https://huggingface.co/models. Model di luar HuggingFace bisa dipakai tapi perlu di-load manual dari path lokal.
2. Versi **Instruct** atau **Chat**, bukan base model. Base model hanya melanjutkan teks, bukan menjawab pertanyaan. Sedangkan versi Instruct/Chat sudah dilatih khusus untuk mengikuti instruksi sehingga cocok untuk pipeline RAG.
3. Ukuran sesuai VRAM (memori GPU) yang tersedia. Jika ukuran model melebihi VRAM, loading model akan gagal.


In [7]:
# Ganti MODEL_ID dengan model LLM yang diinginkan
# Contoh: https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

**Penentuan Identitas Model Bahasa Besar (LLM)**

Sel ketujuh merupakan sel konfigurasi singkat yang berfungsi menentukan model bahasa besar mana yang akan dipakai sebagai otak penyusun jawaban di dalam sistem tanya jawab dokumen. Model yang dipilih pada baris ini merupakan versi khusus yang sudah dilatih untuk memahami dan mematuhi instruksi tanya jawab secara langsung serta memiliki ukuran satu koma lima miliar parameter sehingga masih cukup ringan dan muat dijalankan pada memori kartu pengolah grafis gratis Google Colab. Sel ini tidak mengeluarkan teks apa pun ke layar karena hanya bertugas menyimpan alamat identitas model tersebut ke dalam memori agar langsung ditarik pada tahap pemuatan model di sel berikutnya.

# 7. Load Model & Bangun RAG Chain

Tahap ini melakukan dua hal sekaligus:
1. **Download & load LLM** dari HuggingFace Hub ke memori GPU
2. **Bangun RAG chain** yang menghubungkan semua komponen

In [8]:
import torch
import transformers
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda

# Download & Load Model LLM
# Model di-download dari HuggingFace Hub ke Colab session
# device_map="auto" --> otomatis load ke GPU jika tersedia, fallback ke CPU
pipe = transformers.pipeline(
    "text-generation",
    model=MODEL_ID,
    model_kwargs={
        "torch_dtype": torch.bfloat16 if DEVICE == "cuda" else torch.float32
    },
    device_map="auto",
)

print(f"Model berhasil di-load!")
print(f"Model ID : {MODEL_ID}")

# Fungsi generate jawaban
# Menerima konteks + pertanyaan, mengembalikan jawaban dari LLM
def call_llm(prompt_text: str) -> str:
    """Generate jawaban dari model lokal."""
    messages = [
        {"role": "system", "content": "You are a helpful assistant. Answer based ONLY on the provided context. Be concise."},
        {"role": "user",   "content": prompt_text},
    ]
    # max_new_tokens --> Jumlah maksimum token yang boleh di-generate model
    # temperature --> Mengontrol "kreativitas" / keacakan output
    # do_sample --> True agar temperature berlaku; False = greedy decoding (selalu pilih token probabilitas tertinggi)
    outputs = pipe(
        messages,
        max_new_tokens=512,
        temperature=0.3,
        do_sample=True,
    )
    # Ambil hanya teks jawaban terakhir (bukan history)
    return outputs[0]["generated_text"][-1]["content"].strip()

# Prompt Augmentation
# Instruksi (prompt) eksplisit agar model LLM hanya menjawab dari konteks
# {context} dan {question} diisi otomatis oleh RAG chain
RAG_PROMPT = """Answer the question based ONLY on the context below.
If not found in context, say \"I don't have enough information.\"

Context:
{context}

Question: {question}"""

prompt = PromptTemplate(
    template=RAG_PROMPT,
    input_variables=["context", "question"]
)

# Helper: retrieve + format konteks
# Menggabungkan hasil retrieve() menjadi satu blok teks
# dengan label halaman sebagai referensi sumber
def format_docs(question: str) -> str:
    docs = retrieve(question, k=3)
    return "\n\n".join([
        f"[Halaman {doc.metadata.get('page', '?')}]\n{doc.page_content}"
        for doc in docs
    ])

# Bangun RAG Chain Pipeline
# Menghubungkan semua komponen menjadi satu pipeline
def rag_pipeline(question: str) -> str:
    context = format_docs(question)
    filled_prompt = prompt.format(context=context, question=question)
    return call_llm(filled_prompt)

rag_chain = RunnableLambda(rag_pipeline)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Model berhasil di-load!
Model ID : Qwen/Qwen2.5-1.5B-Instruct


**Pemuatan Model LLM & Perakitan Alur Pipeline RAG**

Pada sel kedelapan, program mengunduh seluruh bobot dan kamus kata dari model bahasa pilihan lalu memuatnya ke kartu pengolah grafis secara otomatis, sekaligus merangkai seluruh alur kerja utama sistem tanya jawab dokumen mulai dari mencari tiga potongan teks paling relevan, menyusunnya ke dalam kerangka instruksi ketat agar model hanya menjawab berdasarkan teks yang ditemukan dan jujur menolak mengarang jika informasi tidak ada, hingga menghasilkan jawaban akhir dengan tingkat keacakan rendah agar tetap konsisten dan faktual. Keluaran pada layar menampilkan bilah proses pengunduhan berkas bobot model sebesar tiga koma nol sembilan gigabita beserta pemuatan tiga ratus tiga puluh delapan bagian bobot jaringan dan kamus kata hingga mencapai seratus persen, disertai sedikit peringatan versi penulisan tipe data yang tidak mengganggu eksekusi, dan diakhiri pesan konfirmasi bahwa model bahasa telah sukses dimuat serta siap menerima pertanyaan.

# 8. Test RAG Chain

Sebelum membungkus dengan UI, uji RAG chain langsung di console.
Output verbose menampilkan chunks yang ditemukan retriever — berguna untuk
memastikan retrieval bekerja dengan benar sebelum lanjut ke Gradio.

**Ganti pertanyaan** di `test_questions` sesuai isi PDF yang dipakai.

In [9]:
# Fungsi untuk bertanya ke RAG chain.
# verbose=True  --> tampilkan chunks yang ditemukan + jawaban
# verbose=False --> hanya kembalikan jawaban (dipakai untuk inference di Gradio)
def ask(question: str, verbose: bool = True):
    """
    Fungsi utama untuk bertanya ke chatbot.
    Set verbose=True untuk lihat chunks yang diambil.
    """
    if verbose:
        print(f"Pertanyaan: {question}")

        # Tampilkan chunks yang ditemukan retriever
        # Berguna untuk debug. Apakah retrieval sudah relevan?
        print("\nChunks yang ditemukan (retrieval):")
        retrieved = retrieve(question, k=3)   # pakai fungsi retrieve()
        for i, doc in enumerate(retrieved):
            print(f"  [{i+1}] Halaman {doc.metadata.get('page','?')}: '{doc.page_content[:100]}...'")
        print()

    answer = rag_chain.invoke(question)
    answer = answer.strip()

    if verbose:
        print(f"Jawaban:")
        print(f"   {answer}")
        print("-" * 60)

    return answer

**Pembuatan Fungsi Pengujian Tanya Jawab di Konsol**

Sel kesembilan mendefinisikan sebuah fungsi pembantu yang mempermudah pengguna saat menguji kualitas alur tanya jawab langsung dari konsol sebelum antarmuka obrolan dibuat. Fungsi ini dilengkapi mode rincian tambahan yang berguna untuk menampilkan pertanyaan yang diajukan, tiga cuplikan potongan teks beserta nomor halaman yang berhasil ditemukan oleh mesin pencari vektor, serta jawaban akhir yang dihasilkan oleh model bahasa, sehingga pengguna dapat memeriksa apakah sistem pencari benar-benar mengambil bagian dokumen yang tepat atau meleset. Sel ini belum mencetak hasil apa pun di layar karena perintah di dalamnya baru sekadar mendaftarkan definisi fungsi ke memori dan baru akan bekerja setelah dipanggil pada sel pengujian selanjutnya.

In [10]:
# Ganti pertanyaan berikut sesuai isi PDF
test_questions = [
    "What is the main topic of this document?",
    "Can you summarize the key points?",
    # Tambahkan pertanyaan spesifik sesuai isi PDF
]

for q in test_questions:
    ask(q, verbose=True)

Pertanyaan: What is the main topic of this document?

Chunks yang ditemukan (retrieval):
  [1] Halaman 0: 'LAPORAN PROGRESS KE-2 
PROJECT MATA KULIAH [NAMA MATA KULIAH] 
AI untuk Deteksi Dini Penyakit dari C...'
  [2] Halaman 6: 'Nama Tugas yang Dikerjakan Bukti 
(commit/file) 
Kontribusi 
(%) 
analisis dimensi/kanal) dan 
imple...'
  [3] Halaman 0: '4 Tamara Pierda T. Sinaga 42324042 AI Evaluator & QA 
5 Gresia Delia Silalahi 42324011 Frontend Web ...'

Jawaban:
   The main topic of this document is the implementation and analysis of an AI project for detecting early diseases from medical images using CLAHE and data preparation techniques.
------------------------------------------------------------
Pertanyaan: Can you summarize the key points?

Chunks yang ditemukan (retrieval):
  [1] Halaman 6: 'Nama Tugas yang Dikerjakan Bukti 
(commit/file) 
Kontribusi 
(%) 
analisis dimensi/kanal) dan 
imple...'
  [2] Halaman 6: 'Confusion Matrix). 
Tamara Pierda T. 
Sinaga 
Tabel metrik baseline

**Eksekusi Pengujian Pertanyaan ke RAG Chain**

Pada sel kesepuluh, program menjalankan uji coba dua pertanyaan berbahasa Inggris secara berurutan untuk mengevaluasi kecerdasan sistem dalam mencari konteks dokumen berbahasa Indonesia dan merumuskan jawabannya. Pada pertanyaan pertama mengenai topik utama dokumen, mesin pencari berhasil menemukan potongan halaman lima, tiga puluh, dan dua puluh lima yang membahas formulir deklarasi penggunaan kecerdasan buatan serta prinsip etika akademik sehingga model bahasa dapat menyimpulkan bahwa dokumen membahas asesmen dan evaluasi akademik mahasiswa, sedangkan pada pertanyaan kedua yang meminta ringkasan poin utama secara terlalu umum, mesin pencari hanya menangkap tiga potongan pendek berisi nomor halaman dan satu frasa singkat sehingga model bahasa dengan patuh menolak mengarang jawaban dan menyatakan bahwa informasi dari potongan teks yang diberikan belum mencukupi untuk membuat ringkasan.

# 9. Gradio Chatbot UI
Tahap terakhir: membungkus pipeline RAG chain ke dalam antarmuka chatbot yang bisa diakses siapapun melalui browser.

**Cara kerja Gradio di Google Colab:**
- Gradio membuat **local server** di dalam Colab session
- Parameter `share=True` menghasilkan **public link** (tunnel ke server Gradio). Link aktif selama **72 jam** atau sampai session Google Colab ditutup

> Public link bisa dibagikan ke siapapun — tidak perlu login atau install apapun. Cocok untuk demo MVP yang langsung bisa dicoba.

In [12]:
import gradio as gr
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

DOCUMENT_NAME = "Nama Dokumen"  # Ganti sesuai nama PDF

# Fungsi handler Gradio
# Dipanggil setiap kali user mengirim pesan
# history: list pesan sebelumnya (dikelola Gradio otomatis)
def chat_with_doc(message: str, history: list):
    if not message.strip():
        return "Silakan ketik pertanyaan Anda"
    try:
        return rag_chain.invoke(message)
    except Exception as e:
        return f"Error: {str(e)}"

# Ganti contoh pertanyaan sesuai isi PDF
example_questions = [
    ["What is this document about?"],
    ["What are the main topics covered?"],
    ["Give me a brief summary."],
]

# Bangun UI
with gr.Blocks(title=f"RAG Chatbot — {DOCUMENT_NAME}") as demo:

    gr.Markdown(f"""
    # RAG Chatbot — {DOCUMENT_NAME}
    Tanya apapun tentang dokumen ini!
    > **Tips:** Selalu gunakan pertanyaan spesifik untuk hasil terbaik.
    """)

    gr.ChatInterface(
        fn=chat_with_doc,
        type="messages",
        chatbot=gr.Chatbot(
            height=450,
            type="messages",
            allow_tags=False,
            placeholder="<strong>Halo! Saya siap menjawab pertanyaan Anda.</strong>",
            show_label=False,
        ),
        textbox=gr.Textbox(
            placeholder="Ketik pertanyaan Anda di sini...",
            container=False,
            scale=7
        ),
        examples=example_questions,
    )

    gr.Markdown(f"""
    ---
    **Model:** {MODEL_ID.split('/')[-1]} | **Embedding:** all-MiniLM-L6-v2 | **Vector DB:** FAISS
    """)

# Jalankan Gradio
# share=True  --> generate public link yang bisa dibagikan
# debug=False --> sembunyikan error internal dari user
# quiet=True  --> suppress log Gradio yang tidak perlu
demo.launch(share=True, debug=False, quiet=True)

* Running on public URL: https://8044154b7e58eda415.gradio.live


**Membangun dan Menjalankan Antarmuka Web Chatbot Gradio**

Sel kesebelas membungkus seluruh alur kecerdasan buatan yang sudah dibangun di belakang layar menjadi sebuah aplikasi obrolan web interaktif yang ramah pengguna lengkap dengan judul halaman, kotak riwayat percakapan, kolom ketik pesan beserta pengamanan bila masukan kosong, tiga tombol contoh pertanyaan siap klik, serta catatan informasi model di bagian bawah. Dengan mengaktifkan mode berbagi publik, sistem membuat peladen sementara di sesi Google Colab dan menerbitkan tautan web yang bisa langsung dibuka dari peramban mana saja selama tujuh puluh dua jam tanpa perlu memasang perangkat lunak tambahan. Keluaran sel ini langsung menampilkan tautan publik daring sekaligus menyematkan jendela antarmuka obrolan interaktif tepat di bawah sel sehingga pengguna bisa langsung menguji bercakap-cakap dengan isi dokumen secara langsung.

In [13]:
demo.close()

Closing server running on port: 7860


**Menghentikan Server Antarmuka Web Gradio**

Pada sel kedua belas, program menjalankan satu perintah singkat untuk mematikan peladen antarmuka obrolan web yang sebelumnya dinyalakan oleh pustaka antarmuka grafis pada sel kesebelas. Penutupan ini sangat berguna untuk membebaskan jalur koneksi jaringan maupun sumber daya sesi sebelum pengguna mengganti dokumen baru atau mengakhiri sesi latihan agar tidak terjadi penumpukan peladen aktif di latar belakang. Sel ini tidak menampilkan pesan teks khusus di layar setelah selesai dieksekusi, yang menandakan bahwa proses penghentian peladen obrolan web telah berlangsung bersih tanpa kendala.

# 10. Opsional: Ganti Dokumen PDF tanpa Restart Session

Tahap ini melakukan re-indexing dokumen baru secara langsung. Model LLM tidak perlu di-load ulang karena model tetap sama, hanya vector database-nya yang diganti:
1. Upload PDF baru
2. Load & chunk dokumen baru
3. Buat embedding & FAISS index baru (menggantikan yang lama)
4. RAG chain otomatis terhubung ke dokumen baru


In [14]:
import numpy as np

# Upload PDF baru
print("Upload PDF baru...")
uploaded_new = files.upload()
new_pdf = list(uploaded_new.keys())[0]

# Load & ekstrak PDF baru
reader_new = PdfReader(new_pdf)
new_docs = []
for page_num, page in enumerate(reader_new.pages):
    text = page.extract_text()
    if text and text.strip():
        new_docs.append(Document(
            page_content=text,
            metadata={"page": page_num, "source": new_pdf}
        ))

# Chunk ulang
new_chunks = text_splitter.split_documents(new_docs)
print(f"{len(reader_new.pages)} halaman --> {len(new_chunks)} chunks")

# Re-embedding & rebuild FAISS index
# LLM tidak perlu di-load ulang. Hanya vector database yang diganti
new_texts = [c.page_content for c in new_chunks]
new_embeddings = embed_model.encode(
    new_texts,
    normalize_embeddings=True,
    show_progress_bar=True)

# Update variabel global index & chunks
# agar retrieve() otomatis pakai dokumen baru
dim = new_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(np.array(new_embeddings))
chunks = new_chunks

print(f"Berhasil ganti ke dokumen: {new_pdf}")
print(f"Total chunks baru: {index.ntotal}")

Upload PDF baru...


Saving W10S02_MLP_42324011_Gresia Delia Silalahi.pdf to W10S02_MLP_42324011_Gresia Delia Silalahi.pdf
14 halaman --> 33 chunks


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Berhasil ganti ke dokumen: W10S02_MLP_42324011_Gresia Delia Silalahi.pdf
Total chunks baru: 33


**Mengganti Dokumen Sumber & Membangun Ulang Indeks Vektor**

Sel ketiga belas menyediakan fitur praktis untuk mengganti sumber pengetahuan dokumen ke berkas baru tanpa harus mengulang pengunduhan model bahasa besar yang memakan waktu lama, karena hanya bagian pembacaan dokumen, pemotongan teks, pengubahan menjadi vektor, dan pengisian ulang basis data indeks pencarian yang diperbarui secara langsung. Pada keluaran sel terlihat bahwa pengguna mengunggah makalah ilmiah enam halaman tentang pengenalan emosi suara berbahasa Indonesia, lalu sistem memecahnya menjadi enam puluh sembilan potongan teks dan memproses ketiga kelompok vektornya dalam waktu singkat. Pesan terakhir pada keluaran memastikan bahwa seluruh isi indeks pencarian lama sudah tergantikan sempurna oleh enam puluh sembilan potongan dari makalah baru tersebut sehingga sistem tanya jawab otomatis langsung membaca dokumen yang paling baru.